In [1]:
import numpy as np
import math
from numba import cuda
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import time

# ==========================================
# 1. THE FORWARD KERNELS (Prediction)
# ==========================================
@cuda.jit
def dense_forward(X, W, b, Z):
    # Grid: (out_features)
    j = cuda.grid(1)
    if j < W.shape[1]:
        s = 0.0
        for i in range(W.shape[0]):
            s += X[i] * W[i, j]
        Z[j] = s + b[j]

@cuda.jit
def relu_forward(Z, A):
    # Grid: (out_features)
    j = cuda.grid(1)
    if j < Z.shape[0]:
        A[j] = Z[j] if Z[j] > 0.0 else 0.0

@cuda.jit
def softmax_forward(Z, A):
    # Single thread execution for sum reduction
    if cuda.grid(1) == 0:
        max_val = -1e9
        for i in range(Z.shape[0]):
            if Z[i] > max_val: 
                max_val = Z[i]
        
        sum_exp = 0.0
        for i in range(Z.shape[0]):
            A[i] = math.exp(Z[i] - max_val)
            sum_exp += A[i]
            
        for i in range(Z.shape[0]):
            A[i] /= sum_exp

# ==========================================
# 2. THE BACKWARD KERNELS (Learning/Calculus)
# ==========================================
@cuda.jit
def cross_entropy_backward(A, y_true, dZ):
    """Calculates the error at the very end of the network."""
    j = cuda.grid(1)
    if j < A.shape[0]:
        target = 1.0 if j == y_true else 0.0
        dZ[j] = A[j] - target

@cuda.jit
def relu_backward(Z, dA, dZ):
    """Passes error backwards through the ReLU."""
    j = cuda.grid(1)
    if j < Z.shape[0]:
        dZ[j] = dA[j] if Z[j] > 0.0 else 0.0

@cuda.jit
def dense_backward_error(W, dZ, dA_prev):
    """Calculates the error to pass to the previous layer."""
    i = cuda.grid(1)
    if i < W.shape[0]:
        s = 0.0
        for j in range(W.shape[1]):
            s += W[i, j] * dZ[j]
        dA_prev[i] = s

@cuda.jit
def dense_backward_update(X, dZ, W, b, lr):
    """Applies Stochastic Gradient Descent (SGD) to weights."""
    # Grid: (in_features, out_features)
    i, j = cuda.grid(2)
    if i < W.shape[0] and j < W.shape[1]:
        # Update Weight
        W[i, j] -= lr * (X[i] * dZ[j])
        
        # Only one thread per column needs to update the bias
        if i == 0:
            b[j] -= lr * dZ[j]

# ==========================================
# 3. DATA LOADING & MEMORY ALLOCATION
# ==========================================
def run_cuda_training():
    print("Downloading MNIST Dataset (This takes a moment)...")
    mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto')
    
    # Normalize pixel values (0 to 1) and convert labels to integers
    X = (mnist.data.values / 255.0).astype(np.float32)
    y = mnist.target.values.astype(np.int32)
    
    # Train/Test Split (Using 10,000 samples for speed. Increase to use full dataset!)
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=10000, test_size=1000, random_state=42)
    
    # --- HYPERPARAMETERS ---
    epochs = 50
    lr = 0.01
    in_feat = 784
    h1 = 128
    h2 = 64
    out_feat = 10

    print("Allocating VRAM and Initializing Weights...")
    # Initialize Weights (He Initialization for ReLU)
    W1 = np.random.randn(in_feat, h1).astype(np.float32) * np.sqrt(2.0/in_feat)
    b1 = np.zeros(h1, dtype=np.float32)
    
    W2 = np.random.randn(h1, h2).astype(np.float32) * np.sqrt(2.0/h1)
    b2 = np.zeros(h2, dtype=np.float32)
    
    W3 = np.random.randn(h2, out_feat).astype(np.float32) * np.sqrt(2.0/h2)
    b3 = np.zeros(out_feat, dtype=np.float32)

    # Move Weights to GPU
    d_W1, d_b1 = cuda.to_device(W1), cuda.to_device(b1)
    d_W2, d_b2 = cuda.to_device(W2), cuda.to_device(b2)
    d_W3, d_b3 = cuda.to_device(W3), cuda.to_device(b3)

    # Pre-allocate intermediate Math Buffers on GPU
    d_X = cuda.device_array(in_feat, dtype=np.float32)
    d_Z1, d_A1 = cuda.device_array(h1, dtype=np.float32), cuda.device_array(h1, dtype=np.float32)
    d_Z2, d_A2 = cuda.device_array(h2, dtype=np.float32), cuda.device_array(h2, dtype=np.float32)
    d_Z3, d_A3 = cuda.device_array(out_feat, dtype=np.float32), cuda.device_array(out_feat, dtype=np.float32)
    
    # Pre-allocate Error (Derivative) Buffers on GPU
    d_dZ3 = cuda.device_array(out_feat, dtype=np.float32)
    d_dA2, d_dZ2 = cuda.device_array(h2, dtype=np.float32), cuda.device_array(h2, dtype=np.float32)
    d_dA1, d_dZ1 = cuda.device_array(h1, dtype=np.float32), cuda.device_array(h1, dtype=np.float32)

    # Calculate Thread Grids
    grid_h1 = math.ceil(h1 / 256)
    grid_h2 = math.ceil(h2 / 256)
    grid_out = math.ceil(out_feat / 32)
    
    grid_w1 = (math.ceil(in_feat/16), math.ceil(h1/16))
    grid_w2 = (math.ceil(h1/16), math.ceil(h2/16))
    grid_w3 = (math.ceil(h2/16), math.ceil(out_feat/16))

    # ==========================================
    # 4. THE TRAINING LOOP
    # ==========================================
    print(f"\n--- Starting 50-Epoch CUDA Training ---")
    
    for epoch in range(epochs):
        start_time = time.time()
        correct_train = 0
        
        for idx in range(len(X_train)):
            # 1. Load one image to GPU
            d_X.copy_to_device(X_train[idx])
            y_true = int(y_train[idx])

            # --- FORWARD PASS ---
            dense_forward[grid_h1, 256](d_X, d_W1, d_b1, d_Z1)
            relu_forward[grid_h1, 256](d_Z1, d_A1)
            
            dense_forward[grid_h2, 256](d_A1, d_W2, d_b2, d_Z2)
            relu_forward[grid_h2, 256](d_Z2, d_A2)
            
            dense_forward[grid_out, 32](d_A2, d_W3, d_b3, d_Z3)
            softmax_forward[1, 1](d_Z3, d_A3)

            # Check training accuracy
            pred = np.argmax(d_A3.copy_to_host())
            if pred == y_true: correct_train += 1

            # --- BACKWARD PASS (Calculus on GPU) ---
            # Layer 3 Error
            cross_entropy_backward[grid_out, 32](d_A3, y_true, d_dZ3)
            dense_backward_error[grid_h2, 256](d_W3, d_dZ3, d_dA2)
            dense_backward_update[grid_w3, (16,16)](d_A2, d_dZ3, d_W3, d_b3, lr)
            
            # Layer 2 Error
            relu_backward[grid_h2, 256](d_Z2, d_dA2, d_dZ2)
            dense_backward_error[grid_h1, 256](d_W2, d_dZ2, d_dA1)
            dense_backward_update[grid_w2, (16,16)](d_A1, d_dZ2, d_W2, d_b2, lr)

            # Layer 1 Error
            relu_backward[grid_h1, 256](d_Z1, d_dA1, d_dZ1)
            dense_backward_update[grid_w1, (16,16)](d_X, d_dZ1, d_W1, d_b1, lr)

        # Epoch Stats
        train_acc = (correct_train / len(X_train)) * 100
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} | Accuracy: {train_acc:.2f}% | Time: {epoch_time:.2f}s")

    print("\nComplete!")

if __name__ == "__main__":
    run_cuda_training()

Allocating VRAM and Initializing Weights...

--- Starting 50-Epoch CUDA Training ---


c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(e

Epoch 1/50 | Accuracy: 86.20% | Time: 7.34s
Epoch 2/50 | Accuracy: 93.95% | Time: 9.62s
Epoch 3/50 | Accuracy: 96.10% | Time: 14.54s
Epoch 4/50 | Accuracy: 97.55% | Time: 12.87s
Epoch 5/50 | Accuracy: 98.50% | Time: 7.15s
Epoch 6/50 | Accuracy: 98.61% | Time: 7.43s
Epoch 7/50 | Accuracy: 99.07% | Time: 7.21s
Epoch 8/50 | Accuracy: 99.41% | Time: 7.27s
Epoch 9/50 | Accuracy: 99.70% | Time: 7.25s
Epoch 10/50 | Accuracy: 99.90% | Time: 7.28s
Epoch 11/50 | Accuracy: 99.99% | Time: 7.24s
Epoch 12/50 | Accuracy: 100.00% | Time: 7.22s
Epoch 13/50 | Accuracy: 100.00% | Time: 7.20s
Epoch 14/50 | Accuracy: 100.00% | Time: 7.20s
Epoch 15/50 | Accuracy: 100.00% | Time: 7.25s
Epoch 16/50 | Accuracy: 100.00% | Time: 7.21s
Epoch 17/50 | Accuracy: 100.00% | Time: 7.21s
Epoch 18/50 | Accuracy: 100.00% | Time: 7.21s
Epoch 19/50 | Accuracy: 100.00% | Time: 7.22s
Epoch 20/50 | Accuracy: 100.00% | Time: 7.24s
Epoch 21/50 | Accuracy: 100.00% | Time: 7.20s
Epoch 22/50 | Accuracy: 100.00% | Time: 7.23s
Epoch 